In [ ]:
# Cell 1: Environment Setup
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Optimizes memory for Windows
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_ID = "meta-llama/Llama-2-7b-hf"

In [ ]:
# Cell 2: 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
# Cell 3: Load Model & Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"  # Flash Attention
)

model = torch.compile(model)
print("Llama 2 is now resident in VRAM. Leave this notebook running.")

In [ ]:
# Cell 4: Load Dataset
from datasets import load_dataset
import os

# 1. UNIVERSAL PATH FIX
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../"))
DATA_PATH = os.path.join(PROJECT_ROOT, "data/processed_training/cleaned/sota_train_bio.jsonl")

# 2. PATH SAFETY CHECK
if os.path.exists(DATA_PATH):
    print(f"Success! Found file at: {DATA_PATH}")
else:
    print(f"ERROR: File not found at {DATA_PATH}")
    print(f"Current Working Directory: {os.getcwd()}")
    if os.path.exists(PROJECT_ROOT):
        print(f"Files in Project Root: {os.listdir(PROJECT_ROOT)}")

# 3. LOAD THE DATASET
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print(f"Ready to train on {len(dataset)} bios.")

In [ ]:
# Cell 5: Configure the Adapter (Rank 64)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the 'parked' model for k-bit training
model = prepare_model_for_kbit_training(model)

# Higher Rank (r=64) captures more facts
peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
print("High-Rank Adapter attached.")

In [ ]:
# Cell 6: The Checkpoint-Enabled Trainer
from trl import SFTTrainer, SFTConfig
import os

# 1. Setup checkpoint directory
CHECKPOINT_DIR = "./models/llama2-bio-checkpoints"

training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=3,           # KEEP ALL 3 EPOCHS
    logging_steps=1,
    bf16=True,

    # --- Checkpoint strategy ---
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,

    packing=True,
    max_seq_length=512,
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

# 2. Resume logic
last_checkpoint = None
if os.path.exists(CHECKPOINT_DIR) and len(os.listdir(CHECKPOINT_DIR)) > 0:
    last_checkpoint = True

trainer.train(resume_from_checkpoint=last_checkpoint)

In [ ]:
# Cell 7: Save & Hub Upload
ADAPTER_PATH = "./models/bio_adapters"

# Save locally first
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print(f"Done! Adapter saved at {ADAPTER_PATH}")